In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Openning data

In [3]:
# tests/preprocessing/datasets.ipynb
from pathlib import Path
import sys, os

# point to your project root
project_root = Path(r"/home/galencarmedeiro/git/postdoc/ragtree")
#project_root = Path(r"C:\Users\henri\Documents\git\post-doc\ragtree")
os.chdir(project_root)  # so relative paths go to data/, not tests/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("CWD:", os.getcwd())

CWD: /home/galencarmedeiro/git/postdoc/ragtree


In [9]:
%run "scripts/run_ontology_linking.py" \
  --dataset-key docred_causal \
  --ontology-key docredontology \
  --method llm_embedding \
  --doc-types dev

[config] dataset-key=docred_causal -> input=data/preprocessed/docred_causal.jsonl
[config] ontology-key=docredontology -> ontology=data/ontology/DocREDOntology/ontology.ttl
[config] method=llm_embedding
[config] output=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl
[run_ontology_linking] Loading ontology from data/ontology/DocREDOntology/ontology.ttl ...
[run_ontology_linking] Building linker method=llm_embedding backend=ollama ...
[run_ontology_linking] Reading data/preprocessed/docred_causal.jsonl and writing /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl ...


106924it [05:41, 312.71it/s]  

[run_ontology_linking] docs_after_type_filter=998 skip=0 limit=None written=998
[run_ontology_linking] Done. Processed 998 documents.


In [13]:
%run "scripts/run_growlrag_relations.py" \
  --dataset-key docred_causal_olink_llm_embedding_onto_docredontology \
  --backend vllm \
  --doc-type-filter dev \
  --growlrag-shot-type dev \
  --growlrag-shot-num 3

[runner] dataset-key=docred_causal_olink_llm_embedding_onto_docredontology
[runner] method=growlrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal_olink_llm_embedding_onto_docredontology.growlrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=['dev']
[runner] skip=0, limit=None
[runner] Loaded DocRED rel_info with 96 entries.


[growlrag] Collecting few-shots (type=dev): 2doc [00:00, 2651.27doc/s]


[growlrag] few-shots: requested=3 collected=3 type=dev shot_skip=0 shot_limit=None


Running growlrag on docred_causal_olink_llm_embedding_onto_docredontology: 998doc [7:13:15, 26.05s/doc]

[runner] Done. Processed 998 documents.
[runner] docs_after_type_filter=998, skip=0, limit=None
[runner] Skipped 0 documents due to doc-type filter.


In [16]:
# baseline (dev)
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method baseline \
  --backend vllm \
  --doc-type dev

# icl (dev)
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method icl \
  --backend vllm \
  --doc-type dev

# cot (dev)
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method cot \
  --backend vllm \
  --doc-type dev


[eval] dataset-key: docred_causal
[eval] method: baseline
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.baseline.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/baseline.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.2139
Recall:    0.1118
F1:        0.1468

=== Counts ===
TP: 1372
FP: 5041
FN: 10903
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.
[eval] dataset-key: docred_causal
[eval] method: icl
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.icl.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/icl.vllm.dev.json

==

In [14]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal_olink_llm_embedding_onto_docredontology \
  --method growlrag \
  --backend vllm \
  --doc-type dev

[eval] dataset-key: docred_causal_olink_llm_embedding_onto_docredontology
[eval] method: growlrag
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal_olink_llm_embedding_onto_docredontology.growlrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal_olink_llm_embedding_onto_docredontology/growlrag.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.2484
Recall:    0.1040
F1:        0.1466

=== Counts ===
TP: 1277
FP: 3864
FN: 10998
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.


In [ ]:
try:
    %run "scripts/run_ontology_linking.py" --dataset-key eventstoryline --ontology-key owltime --method llm_embedding --doc-types all
except Exception as e:
    print(e)

[config] dataset-key=eventstoryline -> input=data/preprocessed/eventstoryline.jsonl
[config] ontology-key=owltime -> ontology=data/ontology/OWLTime/time.ttl
[config] method=llm_embedding
[config] output=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl
[run_ontology_linking] Loading ontology from data/ontology/OWLTime/time.ttl ...
[run_ontology_linking] Building linker method=llm_embedding backend=ollama ...
[run_ontology_linking] Reading data/preprocessed/eventstoryline.jsonl and writing /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl ...


443it [01:19,  5.55it/s]

[run_ontology_linking] docs_after_type_filter=443 skip=0 limit=None written=443
[run_ontology_linking] Done. Processed 443 documents.


In [19]:
try:
    %run "scripts/run_growlrag_relations.py" --dataset-key eventstoryline_olink_llm_embedding_onto_owltime --backend vllm --doc-type-filter all --growlrag-shot-type all --growlrag-shot-num 3
except Exception as e:
    print(e)

[runner] dataset-key=eventstoryline_olink_llm_embedding_onto_owltime
[runner] method=growlrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline_olink_llm_embedding_onto_owltime.growlrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[growlrag] Collecting few-shots (type=all): 443doc [00:00, 8980.42doc/s]


[growlrag] few-shots: requested=3 collected=0 type=all shot_skip=0 shot_limit=None


Running growlrag on eventstoryline_olink_llm_embedding_onto_owltime: 443doc [2:10:18, 17.65s/doc]

[runner] Done. Processed 443 documents.
[runner] docs_after_type_filter=443, skip=0, limit=None


In [20]:
try:
    %run "scripts/eval_relations.py" --dataset-key eventstoryline_olink_llm_embedding_onto_owltime --method growlrag --backend vllm --doc-type all
except Exception as e:
    print(e)

[eval] dataset-key: eventstoryline_olink_llm_embedding_onto_owltime
[eval] method: growlrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline_olink_llm_embedding_onto_owltime.growlrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/eventstoryline_olink_llm_embedding_onto_owltime/growlrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.2493
Recall:    0.0564
F1:        0.0920

=== Counts ===
TP: 544
FP: 1638
FN: 9096
num_docs_seen: 443
num_docs_eval: 443
num_docs_missing_gold: 0

[eval] Done.


###  CausalBank

In [24]:
#try:
#    %run "scripts/run_ontology_linking.py" --dataset-key causalbank --ontology-key wordnetfull --method llm_embedding --doc-types all
#except Exception as e:
#    print(e)

try:
    %run "scripts/run_growlrag_relations.py" --dataset-key causalbank_olink_llm_embedding_onto_wordnetfull --backend vllm --doc-type-filter all --growlrag-shot-type resulted_from --growlrag-shot-num 3
except Exception as e:
    print(e)

try:
    %run "scripts/eval_relations.py" --dataset-key causalbank_olink_llm_embedding_onto_wordnetfull --method growlrag --backend vllm --doc-type all
except Exception as e:
    print(e)

[runner] dataset-key=causalbank_olink_llm_embedding_onto_wordnetfull
[runner] method=growlrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/causalbank_olink_llm_embedding_onto_wordnetfull.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank_olink_llm_embedding_onto_wordnetfull.growlrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[growlrag] Collecting few-shots (type=resulted_from): 2doc [00:00, 7745.71doc/s]


[growlrag] few-shots: requested=3 collected=3 type=resulted_from shot_skip=0 shot_limit=None


Running growlrag on causalbank_olink_llm_embedding_onto_wordnetfull: 1080doc [2:53:15,  9.63s/doc]

[runner] Done. Processed 1080 documents.
[runner] docs_after_type_filter=1080, skip=0, limit=None
[eval] dataset-key: causalbank_olink_llm_embedding_onto_wordnetfull
[eval] method: growlrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/causalbank_olink_llm_embedding_onto_wordnetfull.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank_olink_llm_embedding_onto_wordnetfull.growlrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/causalbank_olink_llm_embedding_onto_wordnetfull/growlrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.8062
Recall:    0.0094
F1:        0.0186

=== Counts ===
TP: 1572
FP: 378
FN: 165538
num_docs_seen: 1080
num_docs_eval: 1080
num_docs_missing_gold: 0

[eval] Done.


### FinCausal

In [25]:
#try:
#    %run "scripts/run_ontology_linking.py" --dataset-key fincausal --ontology-key fibocoreplus --method llm_embedding --doc-types all
#except Exception as e:
#    print(e)

try:
    %run "scripts/run_growlrag_relations.py" --dataset-key fincausal_olink_llm_embedding_onto_fibocoreplus --backend vllm --doc-type-filter all --growlrag-shot-type train.csv --growlrag-shot-num 3
except Exception as e:
    print(e)

try:
    %run "scripts/eval_relations.py" --dataset-key fincausal_olink_llm_embedding_onto_fibocoreplus --method growlrag --backend vllm --doc-type all
except Exception as e:
    print(e)

[runner] dataset-key=fincausal_olink_llm_embedding_onto_fibocoreplus
[runner] method=growlrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/fincausal_olink_llm_embedding_onto_fibocoreplus.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal_olink_llm_embedding_onto_fibocoreplus.growlrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[growlrag] Collecting few-shots (type=train.csv): 12doc [00:00, 35519.86doc/s]


[growlrag] few-shots: requested=3 collected=3 type=train.csv shot_skip=0 shot_limit=None


Running growlrag on fincausal_olink_llm_embedding_onto_fibocoreplus: 0doc [00:00, ?doc/s]

Running growlrag on fincausal_olink_llm_embedding_onto_fibocoreplus: 967doc [45:51,  2.85s/doc]

[runner] Done. Processed 967 documents.
[runner] docs_after_type_filter=967, skip=0, limit=None
[eval] dataset-key: fincausal_olink_llm_embedding_onto_fibocoreplus
[eval] method: growlrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/fincausal_olink_llm_embedding_onto_fibocoreplus.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal_olink_llm_embedding_onto_fibocoreplus.growlrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/fincausal_olink_llm_embedding_onto_fibocoreplus/growlrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.9967
Recall:    0.9849
F1:        0.9908

=== Counts ===
TP: 915
FP: 3
FN: 14
num_docs_seen: 967
num_docs_eval: 967
num_docs_missing_gold: 0

[eval] Done.


### Maven_ERE

In [26]:
#try:
#    %run "scripts/run_ontology_linking.py" --dataset-key maven_ere --ontology-key EventKG --method llm_embedding --doc-types all
#except Exception as e:
#    print(e)

try:
    %run "scripts/run_growlrag_relations.py" --dataset-key maven_ere_olink_llm_embedding_onto_EventKG --backend vllm --doc-type-filter all --growlrag-shot-type train.csv --growlrag-shot-num 3
except Exception as e:
    print(e)

try:
    %run "scripts/eval_relations.py" --dataset-key maven_ere_olink_llm_embedding_onto_EventKG --method growlrag --backend vllm --doc-type all
except Exception as e:
    print(e)

[runner] dataset-key=maven_ere_olink_llm_embedding_onto_EventKG
[runner] method=growlrag
[runner] backend=vllm, model=openai/gpt-oss-20b
[runner] input=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/maven_ere_olink_llm_embedding_onto_EventKG.jsonl
[runner] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere_olink_llm_embedding_onto_EventKG.growlrag.vllm.jsonl
[runner] output-format=full
[runner] doc-type-filter=all
[runner] skip=0, limit=None


[growlrag] Collecting few-shots (type=train.csv): 3516doc [00:00, 17132.48doc/s]


[growlrag] few-shots: requested=3 collected=0 type=train.csv shot_skip=0 shot_limit=None


Running growlrag on maven_ere_olink_llm_embedding_onto_EventKG: 3516doc [23:10:30, 23.73s/doc]


[runner] Done. Processed 3516 documents.
[runner] docs_after_type_filter=3516, skip=0, limit=None
[eval] dataset-key: maven_ere_olink_llm_embedding_onto_EventKG
[eval] method: growlrag
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/maven_ere_olink_llm_embedding_onto_EventKG.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere_olink_llm_embedding_onto_EventKG.growlrag.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/maven_ere_olink_llm_embedding_onto_EventKG/growlrag.vllm.all.json

=== Micro-level metrics ===
Precision: 0.1326
Recall:    0.0394
F1:        0.0608

=== Counts ===
TP: 1815
FP: 11873
FN: 44199
num_docs_seen: 3516
num_docs_eval: 3516
num_docs_missing_gold: 0

[eval] Done.
